In [1]:
from pynq import Overlay
from pynq import allocate
from pynq import MMIO
from pynq import Register
from pynq import Clocks
from cffi import FFI
import numpy as np
import time

#replace the path of acc_finalv2.xsa of your own
ol = Overlay("acc_finalv2.xsa")

In [2]:
#down the bitstream to borad
ol.download()

In [3]:
#print the debug info
ol.is_loaded()
help(ol)
%pwd

Help on Overlay in module pynq.overlay:

<pynq.overlay.Overlay object>
    Default documentation for overlay acc_finalv2.xsa. The following
    attributes are available on this overlay:
    
    IP Blocks
    ----------
    axi_dma_0            : pynq.lib.dma.DMA
    axi_dma_1            : pynq.lib.dma.DMA
    interconnect         : pynq.overlay.DefaultIP
    SysCtrl              : pynq.overlay.DefaultIP
    zynq_ultra_ps_e_0    : pynq.overlay.DefaultIP
    
    Hierarchies
    -----------
    None
    
    Interrupts
    ----------
    None
    
    GPIO Outputs
    ------------
    None
    
    Memories
    ------------
    axi_bram_ctrl_0      : Memory
    PSDDR                : Memory



'/root/jupyter_notebooks/final_v2'

In [4]:
# Read binary file as uint16
data_bin_x = np.fromfile('X_test_tensor_bf16.bin', dtype=np.uint16)
data_bin_y = np.fromfile('Y_test_tensor_bf16.bin', dtype=np.uint16)

In [5]:
# select the function via the parameter[name]. 
# Note the difference of elementwise operation and other operation.

def run_acc(name):
    if(name == b"add" or name == b"mul"):
        elapsed = lib_dma.dma_transfer_two_inputs(
            BRAMX_BASE + 46*768*2,
            BRAMY_BASE + 46*768*2,
            BRAMOut_BASE + 46*768*2,
            (53-46) * 768 * 2,
            ffi.new("char[]", name)
        )
    else:
        elapsed = lib_dma.dma_transfer_one_input(
        BRAMX_BASE,
        BRAMOut_BASE,
        64 * 768 * 2,
        ffi.new("char[]", name)
    )
        
    return elapsed

In [10]:
# modify the freq of the PL clock.
clock = Clocks
clock.fclk0_mhz = 300.0

#reset PL
DIRM5 = Register(0xFF0A0344)
DIRM5[31] = 1

DATA5 = Register(0xFF0A0054)
DATA5[31] = 0

time.sleep(0.5)
DATA5[31] = 1
time.sleep(0.5)


# write the bram.
# Note the input buffer and the output buffer are at the same area
BRAMX_BASE = 0xA4080000
BRAMX_SIZE = 128 * 1024
BRAMY_BASE = 0xA40A0000
BRAMY_SIZE = 128 * 1024
BRAMOut_BASE = 0xA4080000
BRAMOut_SIZE = 128 * 1024

bramX = MMIO(BRAMX_BASE, BRAMX_SIZE)
bramY = MMIO(BRAMY_BASE, BRAMY_SIZE)
bramOut = MMIO(BRAMOut_BASE, BRAMOut_SIZE)

words_X = (data_bin_x[1::2].astype(np.uint32) << 16) | data_bin_x[0::2]
words_Y = (data_bin_y[1::2].astype(np.uint32) << 16) | data_bin_y[0::2]

for i, w in enumerate(words_X):
    bramX.write(i * 4, int(w))

for i, w in enumerate(words_Y):
    bramY.write(i * 4, int(w))


    
# To measure time more precisely and enhance the speed, 
# we use the cffi to import c code. 
ffi = FFI()

ffi.cdef("""
    double dma_transfer_one_input(unsigned int input_addr, unsigned int output_addr, unsigned int length, const char* operation_name);
    double dma_transfer_two_inputs(unsigned int input_addr_x, unsigned int input_addr_y, unsigned int output_addr, unsigned int length, const char* operation_name);
""")
lib_dma = ffi.dlopen("./config.so")

#type the name of your function. eg.softmax, silu, gelu, rmsnorm, layernorm, mul, add
elapsed = run_acc(b"softmax")
# elapsed = run_acc(b"silu")
# elapsed = run_acc(b"gelu")
# elapsed = run_acc(b"rmsnorm")
# elapsed = run_acc(b"layernorm")
# elapsed = run_acc(b"mul")
# elapsed = run_acc(b"add")


print(f"DMA transfer done, took {elapsed:.6f} ns")


DMA transfer done, took 5760.000000 ns


In [11]:
# allocate a buffer to recive the output data.
output_buffer = np.zeros([64, 768]).astype(np.uint16).flatten()

# read the result data from the bram.
for i in range(0, 64*768, 2):
    word = bramOut.read(i * 2)

    low  = word & 0xFFFF
    high = (word >> 16) & 0xFFFF

    output_buffer[i] = low
    output_buffer[i+1] = high

# reshape the buffer to 64*768 for the convenience of inspect
output_buffer = output_buffer.reshape(64, 768)
    
def bf16_to_float32(arr_uint16):
    # 确保是 uint16 格式
    arr_uint16 = np.asarray(arr_uint16, dtype=np.uint16)
    # 左移 16 位，补足 float32 的低位为 0
    arr_uint32 = arr_uint16.astype(np.uint32) << 16
    # 重新解释为 float32
    arr_f32 = arr_uint32.view(np.float32)
    return arr_f32

converted = bf16_to_float32(output_buffer)

# You can watch the results if you want
# Or you can run the next cell to save the result as npy 
print(converted[0,:])


[0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463
 0.00130463 0.00130463 0.00130463 0.00130463 0.00130463 0.0013

In [25]:
np.save("result.npy", output_buffer)